## Purpose: We want to see if invoice page extraction is applicable on all document types. Therefore fetch all document that we did not recognize, and check theirs invoice page predictions

In [30]:
import pandas as pd
from python_utilities.db_connection import DbConnection

analytics_db = DbConnection("ANALYTICS", "PROD_RDS")

INFO [2026-07-21 17:29:54] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file


In [31]:
# All predictions for attachments that were recognized as BOTH
# vermogenverzeichnis (is_va = True) AND drittauskunft (is_dritt = True).
query_va_and_dritt = """
SELECT *
FROM llm_attachments_predictions lap
WHERE lap.created_at >= DATE_SUB(CURDATE(), INTERVAL 30 DAY)
  AND lap.attachment_id IN (
        SELECT attachment_id
        FROM llm_attachments_predictions
        WHERE type = "vermogenverzeichnis_egvp" AND subtype = "is_va" AND value = "'True'"
      )
  AND lap.attachment_id IN (
        SELECT attachment_id
        FROM llm_attachments_predictions
        WHERE type = "drittauskunft_egvp" AND subtype = "is_dritt" AND value = "'True'"
      )
"""

data_va_and_dritt = analytics_db.sql_to_df(query_va_and_dritt)
data_va_and_dritt


,id,created_at,model_name,type,subtype,value,attachment_id
0,11472384,2026-06-24 14:07:43,aftercourt_classification_ladung,aftercourt_classification_ladung,aftercourt_type,'ladung_va',69456918
1,11472385,2026-06-24 14:07:43,aftercourt_classification_ladung,aftercourt_classification_ladung,class_pred,'False',69456918
2,11472386,2026-06-24 14:07:43,aftercourt_classification_ladung,aftercourt_classification_ladung,class_prob,'0.0',69456918
3,11472387,2026-06-24 14:07:43,aftercourt_classification_pfub,aftercourt_classification_pfub,aftercourt_type,'pfub_erlass',69456918
4,11472388,2026-06-24 14:07:43,aftercourt_classification_pfub,aftercourt_classification_pfub,class_pred,'False',69456918
...,...,...,...,...,...,...,...
407,13020825,2026-07-13 12:51:32,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',72081160
408,13020826,2026-07-13 12:51:32,invoice_detection_egvp,invoice_detection_egvp,start_page,'1',72081160
409,13020827,2026-07-13 12:51:32,invoice_detection_egvp,invoice_detection_egvp,end_page,'1',72081160
410,13020828,2026-07-13 12:51:32,egvp_standalone_invoice,egvp_standalone_invoice,invoice,'False',72081160


In [32]:
# All predictions for attachments that were recognized as EITHER
# pfub_erlass_egvp (is_pfub = True) OR aftercourt_classification_ladung (class_pred = True).
query_pfub_or_ladung = """
SELECT *
FROM llm_attachments_predictions lap
WHERE lap.created_at >= DATE_SUB(CURDATE(), INTERVAL 1 DAY)
  AND (
        lap.attachment_id IN (
              SELECT attachment_id
              FROM llm_attachments_predictions
              WHERE type = "pfub_erlass_egvp" AND subtype = "is_pfub" AND value = "'True'"
            )
     OR lap.attachment_id IN (
              SELECT attachment_id
              FROM llm_attachments_predictions
              WHERE type = "aftercourt_classification_ladung" AND subtype = "class_pred" AND value = "'True'"
            )
      )
"""

data_pfub_or_ladung = analytics_db.sql_to_df(query_pfub_or_ladung)
data_pfub_or_ladung


,id,created_at,model_name,type,subtype,value,attachment_id
0,13644148,2026-07-20 05:40:38,aftercourt_classification_ladung,aftercourt_classification_ladung,aftercourt_type,'ladung_va',72611282
1,13644149,2026-07-20 05:40:38,aftercourt_classification_ladung,aftercourt_classification_ladung,class_pred,'False',72611282
2,13644150,2026-07-20 05:40:38,aftercourt_classification_ladung,aftercourt_classification_ladung,class_prob,'0.01',72611282
3,13644151,2026-07-20 05:40:38,aftercourt_classification_pfub,aftercourt_classification_pfub,aftercourt_type,'pfub_erlass',72611282
4,13644152,2026-07-20 05:40:38,aftercourt_classification_pfub,aftercourt_classification_pfub,class_pred,'True',72611282
...,...,...,...,...,...,...,...
13882,13800994,2026-07-21 09:45:40,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'False',72662724
13883,13800995,2026-07-21 09:45:40,invoice_detection_egvp,invoice_detection_egvp,start_page,'-1',72662724
13884,13800996,2026-07-21 09:45:40,invoice_detection_egvp,invoice_detection_egvp,end_page,'-1',72662724
13885,13800997,2026-07-21 09:45:40,egvp_standalone_invoice,egvp_standalone_invoice,invoice,'False',72662724


In [33]:
zendesk = data_pfub_or_ladung[data_pfub_or_ladung['attachment_id'].str.contains("-")]

In [37]:
zendesk

,id,created_at,model_name,type,subtype,value,attachment_id
27,13645484,2026-07-20 06:24:11,aftercourt_classification_ladung,aftercourt_classification_ladung,aftercourt_type,'ladung_va',29021011898780-1
28,13645485,2026-07-20 06:24:11,aftercourt_classification_ladung,aftercourt_classification_ladung,class_pred,'False',29021011898780-1
29,13645486,2026-07-20 06:24:11,aftercourt_classification_ladung,aftercourt_classification_ladung,class_prob,'0.0',29021011898780-1
30,13645487,2026-07-20 06:24:11,aftercourt_classification_pfub,aftercourt_classification_pfub,aftercourt_type,'pfub_erlass',29021011898780-1
31,13645488,2026-07-20 06:24:11,aftercourt_classification_pfub,aftercourt_classification_pfub,class_pred,'True',29021011898780-1
...,...,...,...,...,...,...,...
13295,13800323,2026-07-21 09:44:44,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'False',29050595858588-1
13296,13800324,2026-07-21 09:44:44,invoice_detection_egvp,invoice_detection_egvp,start_page,'-1',29050595858588-1
13297,13800325,2026-07-21 09:44:44,invoice_detection_egvp,invoice_detection_egvp,end_page,'-1',29050595858588-1
13298,13800326,2026-07-21 09:44:44,egvp_standalone_invoice,egvp_standalone_invoice,invoice,'False',29050595858588-1


In [38]:
zendesk.created_at.max()

Timestamp('2026-07-21 09:44:44')

In [39]:
zendesk.created_at.min()

Timestamp('2026-07-20 06:24:11')

In [45]:
zendesk = zendesk[zendesk['created_at'] >= '2026-07-21']

In [46]:
import sys
sys.path.append('/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation')
from utils.prod_utils import pivot_attachment_predictions

zendesk = pivot_attachment_predictions(zendesk)

In [47]:
zendesk

,attachment_id,death_certificate_detection_confidence,death_certificate_detection_is_death_certificate,debt_counseling_attachment_attachment_type,debt_counseling_attachment_casefile_number,debt_counseling_attachment_counselling_slug,debt_counseling_attachment_counsellor_address,debt_counseling_attachment_counsellor_agency_name,debt_counseling_attachment_counsellor_email,debt_counseling_attachment_counsellor_firstname,...,pfub_erlass_egvp_creditor_name,pfub_erlass_egvp_debtor_name,pfub_erlass_egvp_is_invoice_inside,pfub_erlass_egvp_is_pfub,pfub_erlass_egvp_slug,pfub_slug,vermogenverzeichnis_egvp_end_page,vermogenverzeichnis_egvp_is_invoice_inside,vermogenverzeichnis_egvp_is_va,vermogenverzeichnis_egvp_start_page
0,29047115133852-1,none,none,none,none,none,none,none,none,none,...,Liquandum Capital Ii Gmbh,Stemann,False,True,158734390855,none,-1,False,False,-1
1,29047115211164-1,none,none,none,none,none,none,none,none,none,...,Liquandum Capital Ii Gmbh,Kwiek,False,True,170209440602,170209440602,-1,False,False,-1
2,29047132072476-1,none,none,none,none,none,none,none,none,none,...,Liquandum Capital Ii Gmbh,Sophie Kling,False,True,110981277806,none,-1,False,False,-1
3,29047148416796-1,none,none,none,none,none,none,none,none,none,...,Liquandum Capital Ii Gmbh,Philipp Marcel Ciba,False,True,110872898124,none,-1,False,False,-1
4,29047148484380-1,none,none,none,none,none,none,none,none,none,...,Liquandum Capital Ii Gmbh,Niehörster,False,True,123399722321,123399722321,-1,False,False,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,29049584098204-1,low,False,other,[198443112117],none,"Wildermuthstraße 6, 85560 Ebersberg",OGVin Straßer,strasser@gvzentrale.de,N.,...,none,none,none,none,none,none,none,none,none,none
88,29049870735644-1,low,False,other,[180774403182],180774403182,"Am Burgfeld 7, 23568 Lübeck",AG Lübeck,ogv.vwl@gerichtsvollzieher.de,Kay,...,none,none,none,none,none,none,none,none,none,none
89,29050595858588-1,none,none,none,none,none,none,none,none,none,...,Liquandum Capital Ii Gmbh,Mihaela Astanoae,False,True,110948487498,none,-1,False,False,-1
90,29050605429020-1,low,False,other,[197463724603],none,"Gillweg 3, 14193 Berlin",none,m.meyer-hartung@gerichtsvollzieher.de,MEYER-HARTUNG,...,none,none,none,none,none,none,none,none,none,none


In [48]:
zendesk.pfub_erlass_egvp_is_pfub.value_counts()

pfub_erlass_egvp_is_pfub
True     79
none      8
False     5
Name: count, dtype: int64

In [49]:
zendesk.ladung_class_pred.value_counts()

ladung_class_pred
False    79
True     13
Name: count, dtype: int64

In [50]:
pfub = zendesk[zendesk['pfub_erlass_egvp_is_pfub'] == 'True']
ladung = zendesk[zendesk['ladung_class_pred'] == 'True']

In [51]:
pfub

,attachment_id,death_certificate_detection_confidence,death_certificate_detection_is_death_certificate,debt_counseling_attachment_attachment_type,debt_counseling_attachment_casefile_number,debt_counseling_attachment_counselling_slug,debt_counseling_attachment_counsellor_address,debt_counseling_attachment_counsellor_agency_name,debt_counseling_attachment_counsellor_email,debt_counseling_attachment_counsellor_firstname,...,pfub_erlass_egvp_creditor_name,pfub_erlass_egvp_debtor_name,pfub_erlass_egvp_is_invoice_inside,pfub_erlass_egvp_is_pfub,pfub_erlass_egvp_slug,pfub_slug,vermogenverzeichnis_egvp_end_page,vermogenverzeichnis_egvp_is_invoice_inside,vermogenverzeichnis_egvp_is_va,vermogenverzeichnis_egvp_start_page
0,29047115133852-1,none,none,none,none,none,none,none,none,none,...,Liquandum Capital Ii Gmbh,Stemann,False,True,158734390855,none,-1,False,False,-1
1,29047115211164-1,none,none,none,none,none,none,none,none,none,...,Liquandum Capital Ii Gmbh,Kwiek,False,True,170209440602,170209440602,-1,False,False,-1
2,29047132072476-1,none,none,none,none,none,none,none,none,none,...,Liquandum Capital Ii Gmbh,Sophie Kling,False,True,110981277806,none,-1,False,False,-1
3,29047148416796-1,none,none,none,none,none,none,none,none,none,...,Liquandum Capital Ii Gmbh,Philipp Marcel Ciba,False,True,110872898124,none,-1,False,False,-1
4,29047148484380-1,none,none,none,none,none,none,none,none,none,...,Liquandum Capital Ii Gmbh,Niehörster,False,True,123399722321,123399722321,-1,False,False,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,29047780409116-1,none,none,none,none,none,none,none,none,none,...,Liquandum Capital Ii Gmbh,Sarah Klee,False,True,none,none,-1,False,False,-1
82,29047802578332-1,none,none,none,none,none,none,none,none,none,...,Liquandum Capital Ii Gmbh,Gauch,True,True,111321612686,none,2,True,False,1
83,29047822790556-1,none,none,none,none,none,none,none,none,none,...,Liquandum Capital Ii Gmbh,Heinrichs,True,True,110972104449,none,2,True,False,1
84,29047950214044-1,none,none,none,none,none,none,none,none,none,...,Liquandum Capital Ii Gmbh,Brotza,False,True,110976744513,none,-1,False,False,-1


In [23]:
ladung

,attachment_id,death_certificate_detection_confidence,death_certificate_detection_is_death_certificate,debt_counseling_attachment_attachment_type,debt_counseling_attachment_casefile_number,debt_counseling_attachment_counselling_slug,debt_counseling_attachment_counsellor_address,debt_counseling_attachment_counsellor_agency_name,debt_counseling_attachment_counsellor_email,debt_counseling_attachment_counsellor_firstname,...,pfub_erlass_egvp_creditor_name,pfub_erlass_egvp_debtor_name,pfub_erlass_egvp_is_invoice_inside,pfub_erlass_egvp_is_pfub,pfub_erlass_egvp_slug,pfub_slug,vermogenverzeichnis_egvp_end_page,vermogenverzeichnis_egvp_is_invoice_inside,vermogenverzeichnis_egvp_is_va,vermogenverzeichnis_egvp_start_page
27,29021718616092-1,low,False,other,[110162291972],none,none,none,none,none,...,none,none,none,none,none,none,none,none,none,none
28,29022691899164-1,low,False,debt_settlement_plan_offer,"[728, 26]",DR_Il_728/26,none,none,none,none,...,none,none,none,none,none,none,none,none,none,none
29,29024750254108-1,low,False,other,[],none,none,none,none,none,...,none,none,none,none,none,none,none,none,none,none
30,29026185801628-1,low,False,other,[174344543734],none,"Joseph-Fraunhofer-Straße 6, 85276 Pfaffenhofen...",none,gv.meier@gvzentrale.de,Meier,...,none,none,none,none,none,none,none,none,none,none
32,29027125985180-1,low,False,other,[],none,none,none,none,none,...,none,none,none,none,none,none,none,none,none,none
33,29027376431132-2,low,False,debt_settlement_plan_offer,[90226],DR II 902/26,none,none,none,none,...,none,none,none,none,none,none,none,none,none,none
35,29029309340444-1,low,False,other,[172476873564],172476873564,"Breite Straße 101, 26919 Brake",Gerichtsvollzieher,behnken@gerichtsvollzieher.de,Rico,...,none,none,none,none,none,none,none,none,none,none
36,29029515034652-1,low,False,other,[111791877132],none,"Otto-Hahn-Straße 30, 41515 Grevenbroich",none,denise.galle@ag-grevenbroich.nrw.de,D.,...,none,none,none,none,none,none,none,none,none,none
37,29029690393884-1,low,False,other,[160910286022],none,"Joseph-Fraunhofer-Straße 6, 85276 Pfaffenhofen",none,gvin.petra.leppmeier@gerichtsvollzieher.de,Petra,...,none,none,none,none,none,none,none,none,none,none
40,29030631829532-1,low,False,other,[176056295627],none,"Bonner Platz 1/IV, 80803 München",Münchner Bank,gvirlbeck@gmx.de,Sabine,...,none,none,none,none,none,none,none,none,none,none


In [27]:
ladung

,attachment_id,death_certificate_detection_confidence,death_certificate_detection_is_death_certificate,debt_counseling_attachment_attachment_type,debt_counseling_attachment_casefile_number,debt_counseling_attachment_counselling_slug,debt_counseling_attachment_counsellor_address,debt_counseling_attachment_counsellor_agency_name,debt_counseling_attachment_counsellor_email,debt_counseling_attachment_counsellor_firstname,...,pfub_erlass_egvp_creditor_name,pfub_erlass_egvp_debtor_name,pfub_erlass_egvp_is_invoice_inside,pfub_erlass_egvp_is_pfub,pfub_erlass_egvp_slug,pfub_slug,vermogenverzeichnis_egvp_end_page,vermogenverzeichnis_egvp_is_invoice_inside,vermogenverzeichnis_egvp_is_va,vermogenverzeichnis_egvp_start_page
27,29021718616092-1,low,False,other,[110162291972],none,none,none,none,none,...,none,none,none,none,none,none,none,none,none,none
28,29022691899164-1,low,False,debt_settlement_plan_offer,"[728, 26]",DR_Il_728/26,none,none,none,none,...,none,none,none,none,none,none,none,none,none,none
29,29024750254108-1,low,False,other,[],none,none,none,none,none,...,none,none,none,none,none,none,none,none,none,none
30,29026185801628-1,low,False,other,[174344543734],none,"Joseph-Fraunhofer-Straße 6, 85276 Pfaffenhofen...",none,gv.meier@gvzentrale.de,Meier,...,none,none,none,none,none,none,none,none,none,none
32,29027125985180-1,low,False,other,[],none,none,none,none,none,...,none,none,none,none,none,none,none,none,none,none
33,29027376431132-2,low,False,debt_settlement_plan_offer,[90226],DR II 902/26,none,none,none,none,...,none,none,none,none,none,none,none,none,none,none
35,29029309340444-1,low,False,other,[172476873564],172476873564,"Breite Straße 101, 26919 Brake",Gerichtsvollzieher,behnken@gerichtsvollzieher.de,Rico,...,none,none,none,none,none,none,none,none,none,none
36,29029515034652-1,low,False,other,[111791877132],none,"Otto-Hahn-Straße 30, 41515 Grevenbroich",none,denise.galle@ag-grevenbroich.nrw.de,D.,...,none,none,none,none,none,none,none,none,none,none
37,29029690393884-1,low,False,other,[160910286022],none,"Joseph-Fraunhofer-Straße 6, 85276 Pfaffenhofen",none,gvin.petra.leppmeier@gerichtsvollzieher.de,Petra,...,none,none,none,none,none,none,none,none,none,none
40,29030631829532-1,low,False,other,[176056295627],none,"Bonner Platz 1/IV, 80803 München",Münchner Bank,gvirlbeck@gmx.de,Sabine,...,none,none,none,none,none,none,none,none,none,none


In [52]:
pfub.shape

(79, 52)

In [53]:
ladung.shape

(13, 52)

In [55]:
pfub.columns

Index(['attachment_id', 'death_certificate_detection_confidence',
       'death_certificate_detection_is_death_certificate',
       'debt_counseling_attachment_attachment_type',
       'debt_counseling_attachment_casefile_number',
       'debt_counseling_attachment_counselling_slug',
       'debt_counseling_attachment_counsellor_address',
       'debt_counseling_attachment_counsellor_agency_name',
       'debt_counseling_attachment_counsellor_email',
       'debt_counseling_attachment_counsellor_firstname',
       'debt_counseling_attachment_counsellor_lastname',
       'debt_counseling_attachment_debtor_firstname',
       'debt_counseling_attachment_debtor_lastname',
       'debt_counseling_attachment_installment_plan_start_date',
       'drittauskunft_egvp_end_page', 'drittauskunft_egvp_is_dritt',
       'drittauskunft_egvp_is_invoice_inside', 'drittauskunft_egvp_start_page',
       'egvp_standalone_invoice_invoice', 'egvp_standalone_invoice_reason',
       'egvp_standalone_invoice_s

In [62]:
ladung_cols = ['attachment_id', 'ladung_class_pred', 'ladung_debtor_name', 'ladung_judicial_summon_date', 'ladung_slug']
pfub_cols = ['attachment_id', 'pfub_erlass_egvp_is_pfub', 'pfub_erlass_egvp_debtor_name', 'pfub_erlass_egvp_slug', 'pfub_erlass_egvp_creditor_name']

In [63]:
ladung_filtered = ladung[ladung_cols]
pfub_filtered = pfub[pfub_cols]

In [65]:
final_df = pd.concat([ladung_filtered, pfub_filtered], axis=0, ignore_index=True)

In [66]:
final_df

,attachment_id,ladung_class_pred,ladung_debtor_name,ladung_judicial_summon_date,ladung_slug,pfub_erlass_egvp_is_pfub,pfub_erlass_egvp_debtor_name,pfub_erlass_egvp_slug,pfub_erlass_egvp_creditor_name
0,29047164196252-1,True,Marcel Ishorst,19-08-2026,197466153263,NaN,NaN,NaN,NaN
1,29047203295388-1,True,Ali Hassan,12-08-2026,176095754576,NaN,NaN,NaN,NaN
2,29047206452636-1,True,Abdullah Abbas,03-08-2026,195828505394,NaN,NaN,NaN,NaN
3,29047206718492-1,True,Ronny Kohlisch,10-08-2026,110175211165,NaN,NaN,NaN,NaN
4,29047218799260-1,True,Kain Aaron Frohmader,10-08-2026,110425699003,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
87,29047780409116-1,NaN,NaN,NaN,NaN,True,Sarah Klee,none,Liquandum Capital Ii Gmbh
88,29047802578332-1,NaN,NaN,NaN,NaN,True,Gauch,111321612686,Liquandum Capital Ii Gmbh
89,29047822790556-1,NaN,NaN,NaN,NaN,True,Heinrichs,110972104449,Liquandum Capital Ii Gmbh
90,29047950214044-1,NaN,NaN,NaN,NaN,True,Brotza,110976744513,Liquandum Capital Ii Gmbh


In [69]:
a_ids = final_df['attachment_id'].tolist()
id_list = ", ".join(f"'{a_id}'" for a_id in a_ids)
query = f"""
SELECT *
FROM llm_attachments
WHERE attachment_id IN ({id_list})
"""

data = analytics_db.sql_to_df(query)


In [70]:
data

,id,ticket_uuid,attachment_id,zendesk_id,comment_id,status,file_name,file_extension,status_written_at,created_at,s3_key,s3_bucket
0,3927100,8ae83047-776f-50a1-b260-75b130d6bcfb,29047115133852-1,19178185,29047115133852,processed,MJ_00520072026094056.pdf,pdf,2026-07-21 06:29:41,2026-07-21 04:56:03,ocr_source_files/2026-07-21/zendesk_id_1917818...,pair-data-engineering-new
1,3927099,ef38ad5e-32ef-537f-9cd5-45c360dd1c28,29047115211164-1,19178187,29047115211164,processed,MJ_00520072026094058.pdf,pdf,2026-07-21 06:29:43,2026-07-21 04:56:03,ocr_source_files/2026-07-21/zendesk_id_1917818...,pair-data-engineering-new
2,3927104,8e321d54-f9cb-51f5-ad30-99ad0e070444,29047132072476-1,19178190,29047132072476,processed,MJ_00420072026094058.pdf,pdf,2026-07-21 06:31:34,2026-07-21 04:56:08,ocr_source_files/2026-07-21/zendesk_id_1917819...,pair-data-engineering-new
3,3927098,b51ab315-639a-50fc-9442-781272bd489c,29047148416796-1,19178186,29047148416796,processed,MJ_00520072026094057.pdf,pdf,2026-07-21 06:31:36,2026-07-21 04:56:02,ocr_source_files/2026-07-21/zendesk_id_1917818...,pair-data-engineering-new
4,3927101,65244e1b-4c47-54f4-bfeb-57a7f2774192,29047148484380-1,19178188,29047148484380,processed,MJ_00520072026094059.pdf,pdf,2026-07-21 06:31:45,2026-07-21 04:56:08,ocr_source_files/2026-07-21/zendesk_id_1917818...,pair-data-engineering-new
...,...,...,...,...,...,...,...,...,...,...,...,...
87,3928564,4812ddad-39e0-5a1a-ab09-681c9a0ce336,29049584098204-1,19180350,29049584098204,processed,Dokument_87426_14072026_111844.pdf,pdf,2026-07-21 08:37:03,2026-07-21 07:25:55,ocr_source_files/2026-07-21/zendesk_id_1918035...,pair-data-engineering-new
88,3928666,3b1a9abc-bc5d-5993-b93a-c7f555eb37f1,29049870735644-1,19180486,29049870735644,processed,DR-II+092426+Nachr+Mitlg.+an+Glb+%C3%BCber+Ter...,pdf,2026-07-21 09:21:49,2026-07-21 07:36:09,ocr_source_files/2026-07-21/zendesk_id_1918048...,pair-data-engineering-new
89,3928941,1b41af06-bb15-5ca6-87cb-cefcc1ff5fd7,29050595858588-1,19180932,29050595858588,processed,Anschreiben_Pair_Finance_GmbH_Pair_Finance_Gmb...,pdf,2026-07-21 09:44:45,2026-07-21 08:03:01,ocr_source_files/2026-07-21/zendesk_id_1918093...,pair-data-engineering-new
90,3928937,89ff52bc-6d8d-5b3c-842d-f2a6e24d118d,29050605429020-1,19180927,29050605429020,processed,DR-II+071226+Nachr+AS+GLV+Termin+zur+VAK+21.07...,pdf,2026-07-21 09:28:12,2026-07-21 08:02:21,ocr_source_files/2026-07-21/zendesk_id_1918092...,pair-data-engineering-new


In [71]:
final_df_all = final_df.merge(data[['attachment_id', 'zendesk_id', 'comment_id', 'created_at']], on='attachment_id', how='left')

In [72]:
final_df_all.columns

Index(['attachment_id', 'ladung_class_pred', 'ladung_debtor_name',
       'ladung_judicial_summon_date', 'ladung_slug',
       'pfub_erlass_egvp_is_pfub', 'pfub_erlass_egvp_debtor_name',
       'pfub_erlass_egvp_slug', 'pfub_erlass_egvp_creditor_name', 'zendesk_id',
       'comment_id', 'created_at'],
      dtype='object')

In [73]:
column_order = [
    'attachment_id', 'zendesk_id', 'comment_id', 'created_at',
    'ladung_class_pred', 'ladung_debtor_name', 'ladung_judicial_summon_date', 'ladung_slug',
    'pfub_erlass_egvp_is_pfub', 'pfub_erlass_egvp_debtor_name', 'pfub_erlass_egvp_creditor_name', 'pfub_erlass_egvp_slug',
]
final_df_all = final_df_all[column_order]
final_df_all


,attachment_id,zendesk_id,comment_id,created_at,ladung_class_pred,ladung_debtor_name,ladung_judicial_summon_date,ladung_slug,pfub_erlass_egvp_is_pfub,pfub_erlass_egvp_debtor_name,pfub_erlass_egvp_creditor_name,pfub_erlass_egvp_slug
0,29047164196252-1,19178183,29047164196252,2026-07-21 04:55:51,True,Marcel Ishorst,19-08-2026,197466153263,NaN,NaN,NaN,NaN
1,29047203295388-1,19178369,29047203295388,2026-07-21 04:59:22,True,Ali Hassan,12-08-2026,176095754576,NaN,NaN,NaN,NaN
2,29047206452636-1,19178439,29047206452636,2026-07-21 05:00:53,True,Abdullah Abbas,03-08-2026,195828505394,NaN,NaN,NaN,NaN
3,29047206718492-1,19178444,29047206718492,2026-07-21 05:00:59,True,Ronny Kohlisch,10-08-2026,110175211165,NaN,NaN,NaN,NaN
4,29047218799260-1,19178356,29047218799260,2026-07-21 04:59:09,True,Kain Aaron Frohmader,10-08-2026,110425699003,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
87,29047780409116-1,19179091,29047780409116,2026-07-21 05:53:22,NaN,NaN,NaN,NaN,True,Sarah Klee,Liquandum Capital Ii Gmbh,none
88,29047802578332-1,19179123,29047802578332,2026-07-21 05:57:47,NaN,NaN,NaN,NaN,True,Gauch,Liquandum Capital Ii Gmbh,111321612686
89,29047822790556-1,19179124,29047822790556,2026-07-21 05:58:13,NaN,NaN,NaN,NaN,True,Heinrichs,Liquandum Capital Ii Gmbh,110972104449
90,29047950214044-1,19179200,29047950214044,2026-07-21 06:04:28,NaN,NaN,NaN,NaN,True,Brotza,Liquandum Capital Ii Gmbh,110976744513


In [74]:
final_df_all.to_csv("processing_results_21_July_2026.csv", index=False)